# Gemini / Vertex AI (Generative) quick start

This notebook shows the minimal steps to call Gemini (Vertex AI generative models) from your local environment.

Assumptions:
- You have the `gcloud` CLI installed and already authenticated locally.
- You have a Google Cloud project with billing enabled.
- Either Application Default Credentials (ADC) are available (via `gcloud auth application-default login`) or you have a service account JSON and set `GOOGLE_APPLICATION_CREDENTIALS`.

What you'll find below:
1. Commands to enable the API and set up a service account (shell).
2. A runnable Python cell to install dependencies.
3. A Python REST example that uses ADC to call the Vertex AI prediction endpoint.

Replace placeholders (project, location, model) with values from your Vertex AI console.

In [1]:
# Install these packages in the notebook environment if not already installed
# Run this cell (it will call pip). In some environments prefix with ! or use %pip.
import sys
import subprocess

packages = [
    "requests",            # for REST example
    "google-auth",         # for ADC token
    # Optional official client for generative models (if available in your env):
    # "google-generative-ai"  # replace with the exact package name if you want client lib
]

subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages)
print('Installed packages:', packages)

Installed packages: ['requests', 'google-auth']



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip3 install --upgrade pip


In [ ]:
# Minimal REST example using ADC to call Vertex AI Gemini
# Replace only if needed
PROJECT_ID = "lyrical-marker-477423-q8"
LOCATION = "global"  # If you get a location error, try "us-central1"
MODEL = "gemini-1.5-flash"  # Example Gemini model

import google.auth
from google.auth.transport.requests import Request
import requests
import json

# Obtain ADC credentials and an access token (uses gcloud ADC or GOOGLE_APPLICATION_CREDENTIALS)
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
creds.refresh(Request())
access_token = creds.token
print('Access token obtained: length', len(access_token))

# Build request for Gemini generateContent
url = (
    f"https://{LOCATION}-aiplatform.googleapis.com/v1/"
    f"projects/{PROJECT_ID}/locations/{LOCATION}/publishers/google/models/{MODEL}:generateContent"
)
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
}

payload = {
    "contents": [
        {
            "role": "user",
            "parts": [{"text": "Write a short haiku about programming."}],
        }
    ]
}

resp = requests.post(url, headers=headers, data=json.dumps(payload))
print('status', resp.status_code)
try:
    data = resp.json()
    # Attempt to print the first candidate's first text part if present
    text = (
        data.get("candidates", [{}])[0]
        .get("content", {})
        .get("parts", [{}])[0]
        .get("text")
    )
    if text:
        print(text)
    else:
        print(data)
except Exception:
    print(resp.text)

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

## Auth options: ADC vs Service Account

- Application Default Credentials (ADC): simplest for local dev. Use:
  - `gcloud auth application-default login`
  - Code automatically discovers credentials via google-auth.
- Service Account (SA): best for automation and CI.
  - Create SA, grant `roles/aiplatform.user`, download key JSON.
  - Set `GOOGLE_APPLICATION_CREDENTIALS` to point to the JSON file before running code.

Below is an example of creating a service account and setting the env var in zsh. Replace names/paths as needed.

## Using an API Key vs Service Account

You can call certain Gemini endpoints with an API key instead of OAuth2 tokens.

Pros of API Key:
- Quick to prototype.
- Simple header param (`?key=API_KEY`).

Cons / Cautions:
- Less granular control and easier to leak.
- Not suitable for production if you need fine-grained IAM or user impersonation.
- Rotate and restrict (HTTP referrers / IP) where possible.

Service Account (OAuth2) recommended for production & secured environments.

Below is an API key example using the same `generateContent` endpoint. Replace `YOUR_API_KEY` with your actual key (DO NOT commit keys to version control).

In [ ]:
# API Key example for Gemini generateContent
# Paste your API key securely (avoid committing to git)
API_KEY = " "
PROJECT_ID = "lyrical-marker-477423-q8"
LOCATION = "us-central1"
MODEL = "gemini-2.5-flash"

import requests
import json

url = (
    f"https://{LOCATION}-aiplatform.googleapis.com/v1/"
    f"projects/{PROJECT_ID}/locations/{LOCATION}/publishers/google/models/{MODEL}:generateContent"
    f"?key={API_KEY}"
)
headers = {"Content-Type": "application/json"}
payload = {
    "contents": [{
        "role": "user",
        "parts": [{"text": "Give me 3 bullet tips to write clean Python."}],
    }]
}

resp = requests.post(url, headers=headers, data=json.dumps(payload))
print('status', resp.status_code)
try:
    data = resp.json()
    text = (
        data.get("candidates", [{}])[0]
        .get("content", {})
        .get("parts", [{}])[0]
        .get("text")
    )
    print(text or data)
except Exception:
    print(resp.text)

status 200
Here are 3 bullet tips to write clean Python:

*   **Adhere to PEP 8 Style Guide:** This is the bedrock of readable Python. Use consistent indentation (4 spaces), meaningful variable/function names (`snake_case`), follow line length limits (79-99 chars), and use proper spacing around operators. Tools like `flake8` or `Pylint` can help automate checking this. Consistent style makes your code instantly more approachable and understandable.

*   **Embrace Pythonic Constructs:** Leverage list/dict comprehensions, generator expressions, and context managers (`with` statements) to write concise and expressive code. Avoid C-style loops where a more idiomatic Python solution exists. Strive for simplicity and directness in your logic, making the intent clear without excessive boilerplate.

*   **Write Small, Focused Functions/Methods:** Each function or method should ideally do one thing and do it well (Single Responsibility Principle). Keep them short, with clear input/output and mi